# Librerías

In [2]:
import gymnasium as gym # Importa la librería Gymnasium, que se usa para crear y manejar entornos de aprendizaje por refuerzo
import numpy as np # Importa NumPy, una librería para operaciones matemáticas y manejo eficiente de arreglos
from random import randint # Importa la función randint del módulo random para generar números enteros aleatorios
# Configurar NumPy para no usar notación científica al imprimir
np.set_printoptions(suppress=True)

# Ambiente

[Lunar Lander](https://gymnasium.farama.org/environments/box2d/lunar_lander/)

![Lunar Lander](https://gymnasium.farama.org/_images/lunar_lander.gif)

Este entorno representa un problema clásico de optimización de la trayectoria de un cohete. Según el principio del máximo de Pontryagin, lo óptimo es encender el motor a máxima potencia o apagarlo. Por ello, este entorno presenta **acciones discretas:** motor encendido o apagado.

Existen dos versiones del entorno: discreta o continua. La plataforma de aterrizaje siempre se encuentra en las coordenadas (0,0). Estas coordenadas corresponden a los dos primeros números del vector de estado. Es posible aterrizar fuera de la plataforma. El combustible es infinito, por lo que un agente puede aprender a volar y aterrizar en su primer intento.

In [7]:
env= gym.make("LunarLander-v3")
env

C:\Users\rsantos\AppData\Roaming\Python\Python312\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


<TimeLimit<OrderEnforcing<PassiveEnvChecker<LunarLander<LunarLander-v3>>>>>

**Objetivo del MDP**

- Lograr que la nave aterrice suavemente sobre la plataforma de aterrizaje.

- El agente debe controlar los motores (principal y laterales) para:
  - Reducir la velocidad de descenso.
  - Mantener una orientación estable.
  - Ajustar la posición horizontal para alinearse con la zona de aterrizaje.

- Requiere control estratégico: el agente debe balancear entre:
  - No caer demasiado rápido (usar motor principal).
  - No desviarse lateralmente (usar motores laterales).
  - No gastar demasiado combustible innecesariamente.

- La recompensa típica:
  - **Positiva** por acercarse al centro de la plataforma.
  - **Positiva** por aterrizar suavemente.
  - **Negativa** por:
    - Alta velocidad al impactar.
    - Mala orientación (inclinación).
    - Uso excesivo de motores (consumo de combustible).
  - **Gran recompensa positiva** si aterriza correctamente.
  - **Gran penalización** si se estrella.

- El episodio termina cuando:
  1. La nave aterriza exitosamente (ambas patas tocan el suelo).
  2. La nave se estrella.
  3. La nave se sale de los límites del entorno.
  4. Se alcanza un número máximo de iteraciones.

**Estados (State Space)**

La observación es un `ndarray` con forma `(8,)` que describe completamente el estado de la nave.


| Número | Observación                      | Mínimo | Máximo |
|--------|----------------------------------|--------|--------|
| 0      | Posición horizontal $x$         | -2.5   | 2.5    |
| 1      | Posición vertical $y$           | -2.5   | 2.5    |
| 2      | Velocidad horizontal $v_x$      | -10.0  | 10.0   |
| 3      | Velocidad vertical $v_y$        | -10.0  | 10.0   |
| 4      | Ángulo $\theta$                | -2$\pi$ | 2$\pi$ |
| 5      | Velocidad angular $\omega$     | -10.0  | 10.0   |
| 6      | Contacto pierna izquierda       | 0      | 1      |
| 7      | Contacto pierna derecha         | 0      | 1      |

*Interpretación*

- $x, y$ → Posición de la nave respecto al centro de la plataforma.
- $v_x, v_y$ → Qué tan rápido se mueve en cada dirección.
- $\theta$ → Inclinación de la nave (idealmente cercana a 0).
- $\omega$ → Qué tan rápido está rotando.
- Contactos:
  - `0` → la pierna no toca el suelo
  - `1` → la pierna está en contacto con el suelo

**Nota importante**

- Los valores mínimos y máximos son aproximados.
- En la práctica, el entorno puede generar valores fuera de estos rangos.
- Por eso, si discretizas, es recomendable usar **clipping**.

In [12]:
low = env.observation_space.low
high = env.observation_space.high

# Extraemos cada componente del estado (8 variables)
x_min, y_min, vx_min, vy_min, ang_min, vang_min, leg1_min, leg2_min = low
x_max, y_max, vx_max, vy_max, ang_max, vang_max, leg1_max, leg2_max = high

print("Posición x            ∈ [{}, {}]".format(x_min, x_max))
print("Posición y            ∈ [{}, {}]".format(y_min, y_max))
print("Velocidad x           ∈ [{}, {}]".format(vx_min, vx_max))
print("Velocidad y           ∈ [{}, {}]".format(vy_min, vy_max))
print("Ángulo θ              ∈ [{}, {}]".format(ang_min, ang_max))
print("Velocidad angular ω   ∈ [{}, {}]".format(vang_min, vang_max))
print("Contacto pierna izq   ∈ [{}, {}]".format(leg1_min, leg1_max))
print("Contacto pierna der   ∈ [{}, {}]".format(leg2_min, leg2_max))

Posición x            ∈ [-2.5, 2.5]
Posición y            ∈ [-2.5, 2.5]
Velocidad x           ∈ [-10.0, 10.0]
Velocidad y           ∈ [-10.0, 10.0]
Ángulo θ              ∈ [-6.2831854820251465, 6.2831854820251465]
Velocidad angular ω   ∈ [-10.0, 10.0]
Contacto pierna izq   ∈ [-0.0, 1.0]
Contacto pierna der   ∈ [-0.0, 1.0]


In [13]:
# ==============================
# INFORMACIÓN DEL ESPACIO DE ESTADOS (LunarLander)
# ==============================

# Imprime la descripción completa del espacio de observaciones
# En LunarLander normalmente verás algo como:
# Box(low, high, (8,), float32)
# → Box: espacio continuo
# → (8,): 8 variables de estado
print(env.observation_space)

# Imprime los valores mínimos posibles para cada dimensión del estado
# Orden:
# [x, y, vx, vy, θ, ω, pierna_izq, pierna_der]
print("Low:", env.observation_space.low)

# Imprime los valores máximos posibles para cada dimensión del estado
print("High:", env.observation_space.high)

# Imprime la forma del estado
# (8,) significa que cada estado tiene 8 variables
print("Shape:", env.observation_space.shape)

# Imprime el tipo de dato
# float32 → números decimales de precisión simple
print("Dtype:", env.observation_space.dtype)

Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Low: [ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ]
High: [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ]
Shape: (8,)
Dtype: float32


**Acciones (Action Space) – Valores Discretos**

El espacio de acciones es **discreto** y representa los motores que puede activar la nave:

- $a \in \{0,1,2,3\}$

  - $0$ → No hacer nada
  - $1$ → Encender motor izquierdo (moverse a la derecha)
  - $2$ → Encender motor principal (impulso hacia arriba)
  - $3$ → Encender motor derecho (moverse a la izquierda)




| Acción | Descripción                          | Tipo      |
|--------|--------------------------------------|----------|
| 0      | No hacer nada                        | Discreta |
| 1      | Motor izquierdo (empuje lateral)     | Discreta |
| 2      | Motor principal (empuje vertical)    | Discreta |
| 3      | Motor derecho (empuje lateral)       | Discreta |

Imprime la descripción completa del espacio de **acciones** del entorno.

Esto nos dice el tipo de espacio (Discrete, Box, etc.), los valores mínimos y máximos

In [17]:
print(env.action_space)

Discrete(4)


Verificamos si el espacio de acciones es discreto o continuo

In [19]:
# Espacios discretos representan acciones como enteros: 0, 1, 2, ...
if isinstance(env.action_space, gym.spaces.Discrete):
    # Imprime el número total de acciones posibles
    print("Número de acciones discretas:", env.action_space.n)
    # Imprime la lista de acciones posibles (enteros)
    print("Acciones posibles:", list(range(env.action_space.n)))
else:
    # Para espacios continuos (Box), imprimimos los rangos mínimos y máximos
    # Esto representa el valor mínimo y máximo que se puede aplicar para cada acción
    print("Acciones continuas - mínimo:", env.action_space.low)
    print("Acciones continuas - máximo:", env.action_space.high)
    # También podemos mostrar la forma y tipo de datos
    print("Shape del espacio de acciones:", env.action_space.shape)
    print("Tipo de datos:", env.action_space.dtype)

Número de acciones discretas: 4
Acciones posibles: [0, 1, 2, 3]


La función  <code> discretizar </code> normaliza el estado 'valor' usando los límites del espacio de observaciones
$$\frac{\textit{valor} - \textit{mínimo}}{\textit{máximo} - \textit{mínimo}}$$
- Cada dimensión queda entre $0$ y $1$.

In [21]:
def discretizar(valor):
    aux = ((valor - env.observation_space.low) / 
           (env.observation_space.high - env.observation_space.low)) * 4
    # Multiplicamos por 20 para crear 20 intervalos discretos por dimensión

    # Convertimos los valores a enteros (truncando decimales)
    aux = aux.astype(np.int32)

    # Devolvemos una tupla de enteros
    # Esto permite usarla como índice en una tabla Q o diccionario
    return tuple(aux)

En **LunarLander-v2**, la discretización se utiliza porque **los estados del entorno son continuos**. Por ejemplo, el espacio de estados incluye:

- `x`: posición horizontal del módulo
- `y`: posición vertical del módulo
- `v_x`: velocidad horizontal
- `v_y`: velocidad vertical
- `theta`: ángulo de orientación
- `omega`: velocidad angular
- `leg_contact_left` y `leg_contact_right`: contacto de las patas (binario)

Dado que la mayoría de estos valores son **continuos**, no es posible usar directamente una **Q-table**, ya que esto requeriría un número infinito de combinaciones de estados. 

Por ello, se recurre a:

- **Discretización del espacio de estados** (dividir rangos continuos en "bins")  
- **Aproximaciones por función** (como redes neuronales)  

Esto permite aplicar métodos basados en **Q-learning** incluso en entornos con estados continuos.

Al discretizar, se convierten estos valores continuos en un número finito de “intervalos” o estados. Esto permite representar el problema con una tabla finita y aplicar Q-learning de forma práctica.

Además, reduce la complejidad del problema y facilita el aprendizaje del agente, ya que agrupa estados similares. Sin embargo, existe un equilibrio: usar pocos intervalos simplifica el aprendizaje pero pierde precisión, mientras que usar muchos mejora la precisión pero incrementa el tiempo de entrenamiento y el tamaño de la Q-table.


**Estado inicial**

En **LunarLander-v2**, el estado inicial del módulo lunar se asigna típicamente de manera aleatoria dentro de ciertos rangos:

- La posición horizontal `x` se inicializa generalmente cerca del centro de la pantalla (por ejemplo, $x \sim \mathcal{U}(-0.1, 0.1)$).
- La posición vertical `y` se inicializa cerca del suelo o a una altura inicial determinada (por ejemplo, $y \sim \mathcal{U}(0.4, 0.6)$ en unidades normalizadas).
- Las velocidades `v_x` y `v_y` se asignan típicamente a `0` o a un valor pequeño cercano a cero.
- El ángulo de orientación `theta` se inicializa generalmente cerca de `0` (vertical) con una pequeña aleatoriedad.
- La velocidad angular `omega` se asigna típicamente a `0`.
- El contacto de las patas (`leg_contact_left` y `leg_contact_right`) se inicializa en `False`.

**Valor máximo al realizar discretización**

In [26]:
estado_inicial_max = np.array([ x_max,   # Posición x
    y_max,   # Posición y
    vx_max,  # Velocidad x
    vy_max,  # Velocidad y
    ang_max, # Ángulo θ
    vang_max,# Velocidad angular ω
    leg1_max,# Contacto pierna izquierda
    leg2_max # Contacto pierna derecha]) #env.reset()[0]
                              ])
estado_inicial_max

array([ 2.5      ,  2.5      , 10.       , 10.       ,  6.2831855,
       10.       ,  1.       ,  1.       ], dtype=float32)

In [27]:
discretizar(estado_inicial_max) #Estado inicial

(4, 4, 4, 4, 4, 4, 4, 4)

**Valor aleatorio al realizar discretización**

In [29]:
estado_inicial =  env.reset()[0]
estado_inicial

array([ 0.00225477,  1.4108154 ,  0.2283676 , -0.00465912, -0.00260591,
       -0.05172865,  0.        ,  0.        ], dtype=float32)

In [30]:
discretizar(estado_inicial) #Estado inicial

(2, 3, 2, 1, 1, 1, 0, 0)

**Valor mínimo al realizar discretización**

In [32]:
estado_inicial_min = np.array([
x_min,   # Posición x
    y_min,   # Posición y
    vx_min,  # Velocidad x
    vy_min,  # Velocidad y
    ang_min, # Ángulo θ
    vang_min,# Velocidad angular ω
    leg1_min,# Contacto pierna izquierda
    leg2_min # Contacto pierna derecha
]) #env.reset()[0]
estado_inicial_min

array([ -2.5      ,  -2.5      , -10.       , -10.       ,  -6.2831855,
       -10.       ,  -0.       ,  -0.       ], dtype=float32)

In [33]:
discretizar(estado_inicial_min) #Estado inicial

(0, 0, 0, 0, 0, 0, 0, 0)

# Modelo

Crear la Q-table para un entorno con **8 dimensiones discretizadas** (`x`, `y`, `v_x`, `v_y`, `theta`, `omega`, `leg_contact_left`, `leg_contact_right`) y **acciones posibles**.

- Cada celda `q_table[i, j, k, l, m, n, o, p, a]` representa la estimación de la recompensa esperada para el estado discreto `(i, j, k, l, m, n, o, p)` al tomar la acción `a`.

Si quieres hacerlo más explícito:

- `i` → índice para la posición horizontal `x`  
- `j` → índice para la posición vertical `y`  
- `k` → índice para la velocidad horizontal `v_x`  
- `l` → índice para la velocidad vertical `v_y`  
- `m` → índice para el ángulo `theta`  
- `n` → índice para la velocidad angular `omega`  
- `o` → índice para el contacto de la pierna izquierda `leg_contact_left`  
- `p` → índice para el contacto de la pierna derecha `leg_contact_right`  
- `a` → acción tomada (`0: no-op`, `1: motor principal`, `2: motor izquierdo`, `3: motor derecho`)  

Creamos la Q-table con valores iniciales aleatorios entre `-1` y `1`.

Dimensiones: `[5, 5, 5, 5, 5, 5, 5, 5, 4]`

- `5` divisiones para la posición horizontal `x` (índices 0 a 4)  
- `5` divisiones para la posición vertical `y` (índices 0 a 4)  
- `5` divisiones para la velocidad horizontal `v_x` (índices 0 a 4)  
- `5` divisiones para la velocidad vertical `v_y` (índices 0 a 4)  
- `5` divisiones para el ángulo `theta` (índices 0 a 4)  
- `5` divisiones para la velocidad angular `omega` (índices 0 a 4)  
- `5` divisiones para el contacto pierna izquierda `leg_contact_left` (0 a  4)  
- `5` divisiones para el contacto pierna derecha `leg_contact_right` (0 a 4)  
- `4` → número de acciones posibles (`0: no-op`, `1: motor principal`, `2: motor izquierdo`, `3: motor derecho`)  

In [37]:
q_table = np.random.uniform(low=-1, high=1, size=[5,5,5,5,5,5,5,5, 4])

In [38]:
len(q_table)

5

## Entrenamiento

## Parámetros de Q-learning

In [41]:
alfa = 0.1        # Tasa de aprendizaje: qué tanto se actualiza la Q-table en cada iteración
gamma = 0.95      # Factor de descuento: qué tanto importan las recompensas futuras
episodios = 15000  # Número total de episodios de entrenamiento
epsilon = 2       # Parámetro de exploración (controla aleatoriedad en acciones)
lista_recompenzas = [] # Lista para guardar la recompensa total obtenida en cada episodio

## Bucle principal de entrenamiento

In [43]:
for episodio in range(episodios):

    # Reiniciamos el entorno y discretizamos el estado inicial
    estado = discretizar(env.reset()[0])

    # Variables de control del episodio
    final = False
    f1 = False
    f2 = False
    
    recompensa_total = 0

    # ==============================
    # INTERACCIÓN CON EL ENTORNO
    # ==============================
    while not final:

        # Política ε-greedy
        valor_random = randint(0,10)
        #print(f"\nEpisodio: {episodio} | Valor random: {valor_random}")

        if valor_random > epsilon:
            accion = np.argmax(q_table[estado])
            tipo = "EXPLOTACIÓN"
        else:
            accion = randint(0,3) #{0,1,2,3}
            tipo = "EXPLORACIÓN"

        #print(f"Tipo: {tipo}, Acción: {accion}, Estado actual: {estado}")

        # Ejecutamos la acción
        nuevo_estado, recompensa, f1, f2, info = env.step(accion)

        #print(f"nuevo_estado: {nuevo_estado}, recompensa: {recompensa}, f1: {f1}, f2: {f2}, info: {info}")

        # ==============================
        # ACTUALIZACIÓN Q-LEARNING
        # ==============================
        estado_discreto_nuevo = discretizar(nuevo_estado)

        valor_actual = q_table[estado][accion]
        max_futuro = np.max(q_table[estado_discreto_nuevo])

        #print(f"Q actual: {valor_actual}, max Q futuro: {max_futuro}")

        # Fórmula Q-learning
        q_table[estado][accion] = (
            valor_actual +
            alfa * (recompensa + gamma * max_futuro - valor_actual)
        )

        #print(f"Q actualizado: {q_table[estado][accion]}")

        # Actualizamos estado
        estado = estado_discreto_nuevo

        # Acumulamos recompensa
        recompensa_total += recompensa

        # Verificamos fin
        final = f1 or f2

        #print(f"Recompensa acumulada: {recompensa_total}, Final: {final}")
        #print("--------------------------------------------------")

    # Guardamos recompensa del episodio
    lista_recompenzas.append(recompensa_total)

    # Print cada 100 episodios
    if (episodio + 1) % 100 == 0:
        print("Episodio: " + str(episodio) +
              " | Recompensa promedio: " + str(np.mean(lista_recompenzas)))

# Cerramos entorno
env.close()

Episodio: 99 | Recompensa promedio: -114.1430967570692
Episodio: 199 | Recompensa promedio: -113.992431350388
Episodio: 299 | Recompensa promedio: -118.8636754723367
Episodio: 399 | Recompensa promedio: -115.71791373813805
Episodio: 499 | Recompensa promedio: -113.24356237471173
Episodio: 599 | Recompensa promedio: -114.74877589204179
Episodio: 699 | Recompensa promedio: -114.05738684420031
Episodio: 799 | Recompensa promedio: -115.96271549789911
Episodio: 899 | Recompensa promedio: -115.1755488626977
Episodio: 999 | Recompensa promedio: -116.15958717547146
Episodio: 1099 | Recompensa promedio: -114.59425151304943
Episodio: 1199 | Recompensa promedio: -117.32007252572802
Episodio: 1299 | Recompensa promedio: -116.60753043885244
Episodio: 1399 | Recompensa promedio: -115.62487412042188
Episodio: 1499 | Recompensa promedio: -115.42240725892287
Episodio: 1599 | Recompensa promedio: -115.31717330682271
Episodio: 1699 | Recompensa promedio: -114.85367726675564
Episodio: 1799 | Recompensa pr

Crear el entorno de Lunar - Lander
- <code> render_mode="human" </code> permite ver la animación en pantalla 

In [140]:
env = gym.make("LunarLander-v3", render_mode="human")

In [142]:
print("Estado:", estado, "Tipo:", type(estado))
print("q_table shape:", q_table.shape)
print("q_table[estado]:", q_table[estado])

Estado: (1, 1, 1, 1, 2, 1, 4, 4) Tipo: <class 'tuple'>
q_table shape: (5, 5, 5, 5, 5, 5, 5, 5, 4)
q_table[estado]: [-27.73200954 -35.72395274 -31.45942833 -42.29832824]


In [144]:
print(f"Estado: {estado}, len: {len(estado)}")

Estado: (1, 1, 1, 1, 2, 1, 4, 4), len: 8


In [146]:
# Reinicia el entorno y obtiene el estado inicial (continuo)
# Luego se discretiza para poder usar la Q-table
estado = discretizar(env.reset()[0])

# Variables de control del episodio
final = False   # Indica si el episodio terminó
f1 = False      # Bandera de éxito (llegó a la meta)
f2 = False      # Bandera de fallo (se terminó el tiempo)


# Mientras el episodio no termine
while not final:
    
    # Elige la mejor acción según la Q-table (explotación)
    accion = np.argmax(q_table[estado])
      
    # Ejecuta la acción en el entorno
    # Devuelve el nuevo estado, la recompensa y si terminó el episodio
    nuevo_estado, recompensa, f1, f2, info = env.step(accion)

    # Convierte el nuevo estado continuo a discreto
    estado = discretizar(nuevo_estado)
  
    # Verifica si el episodio terminó (éxito o fallo)
    final = f1 or f2

# Cierra el entorno
env.close()